# Benchmark – Predikce výnosu plodin v Indii

Tento notebook vytváří jednoduchý benchmark. Modely se porovnávají pouze na validační sadě. Test set se nepoužívá.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge

DATA = Path('data')
train = pd.read_csv(DATA / 'train.csv', low_memory=False)
validation = pd.read_csv(DATA / 'validation.csv', low_memory=False)

print('train:', train.shape)
print('validation:', validation.shape)
display(train.head())


train: (143177, 23)
validation: (21349, 23)


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield,rain_sum_mm,rainy_days_ge1mm,...,heavy_rain_days_ge20mm,temp_mean_c,temp_max_mean_c,temp_min_mean_c,hot_days_tmax_ge35c,humidity_mean_pct,solar_mean,planting_dry_flag,harvest_too_wet_flag,lag_yield
0,Andaman and Nicobar Islands,NICOBARS,2010,Autumn,Rice,3.5,10.00,2.857143,726.47,87.0,...,5.0,27.58,27.88,27.22,0.0,81.98,14.75,0.0,1.0,NaN
1,Andaman and Nicobar Islands,NICOBARS,2010,Autumn,Rice,3.5,1000.00,285.714286,726.47,87.0,...,5.0,27.58,27.88,27.22,0.0,81.98,14.75,0.0,1.0,2.857143
2,Andaman and Nicobar Islands,NICOBARS,2010,Autumn,Sugarcane,13.4,41.75,3.115672,726.47,87.0,...,5.0,27.58,27.88,27.22,0.0,81.98,14.75,0.0,1.0,NaN
3,Andaman and Nicobar Islands,NICOBARS,2010,Autumn,Sugarcane,13.4,4175.00,311.567164,726.47,87.0,...,5.0,27.58,27.88,27.22,0.0,81.98,14.75,0.0,1.0,3.115672
4,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.00,1.594896,978.01,133.0,...,9.0,27.62,27.86,27.31,0.0,83.30,17.02,0.0,1.0,NaN


## 1. Metriky

Používám MAE, RMSE a R². Hlavní metrika pro interpretaci je MAE.

In [2]:
def evaluate(y_true, y_pred, name):
    return {
        'model': name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': r2_score(y_true, y_pred),
    }

results = []
y_train = train['yield']
y_val = validation['yield']


## 2. Naivní baseline

In [3]:
results.append(evaluate(y_val, np.repeat(y_train.mean(), len(validation)), 'Naive baseline: train mean'))
results.append(evaluate(y_val, np.repeat(y_train.median(), len(validation)), 'Naive baseline: train median'))


## 3. Historický baseline přes lag_yield

In [4]:
lag_pred = validation['lag_yield'].fillna(y_train.median())
results.append(evaluate(y_val, lag_pred, 'Historical baseline: lag_yield'))


## 4. Jednoduché ML benchmarky

`Production` záměrně není mezi features, protože by způsobovala leakage.

In [5]:
weather_cols = [
    'rain_sum_mm', 'rainy_days_ge1mm', 'dry_days_lt1mm',
    'longest_dry_spell_days', 'longest_wet_spell_days',
    'heavy_rain_days_ge20mm', 'temp_mean_c', 'temp_max_mean_c',
    'temp_min_mean_c', 'hot_days_tmax_ge35c', 'humidity_mean_pct',
    'solar_mean', 'planting_dry_flag', 'harvest_too_wet_flag'
]

numeric_features = ['Crop_Year', 'Area', 'lag_yield'] + [c for c in weather_cols if c in train.columns]
categorical_features = ['State_Name', 'District_Name', 'Season', 'Crop']
feature_cols = numeric_features + categorical_features

X_train = train[feature_cols].copy()
X_val = validation[feature_cols].copy()

try:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse=True)

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler(with_mean=False)),
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', onehot),
        ]), categorical_features),
    ]
)

models = {
    'Ridge regression': Ridge(alpha=1.0, random_state=42),
    'Decision tree regressor': DecisionTreeRegressor(max_depth=12, min_samples_leaf=20, random_state=42),
}

for name, model in models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    results.append(evaluate(y_val, pred, name))


## 5. Vyhodnocení na validační sadě

In [6]:
results_df = pd.DataFrame(results).sort_values('MAE')
display(results_df)
results_df.to_csv(DATA / 'benchmark_validation_results.csv', index=False)


,model,MAE,RMSE,R2
4,Decision tree regressor,4637.201851,54039.434574,0.634475
1,Naive baseline: train median,7341.051489,89682.094756,-0.006716
3,Ridge regression,9702.029454,63472.090582,0.495732
2,Historical baseline: lag_yield,10982.934572,105746.298349,-0.399671
0,Naive baseline: train mean,12523.881113,89397.541384,-0.000338


## 6. Shrnutí

Benchmark obsahuje naivní baseline, historický baseline a dva jednoduché ML modely. Test set nebyl použit.